In [0]:
# Check the cluster
# v2 
attached_cluster_name = spark.conf.get(
    "spark.databricks.clusterUsageTags.clusterName", ""
)
if not attached_cluster_name.endswith("uc_support") and not (
    attached_cluster_name.startswith("bfdw_")
    and "compute_uc_jobs" in attached_cluster_name
):
    raise Exception(
        "This notebook is being executed in an incorrect cluster. Please attach it to the *uc_support cluster or one of the bfdw_*compute_uc_jobs* clusters"
    )
else:
    print(f"Cluster is: {attached_cluster_name}")

2.Code to select catolog based on the workspace

In [0]:
workspace_catalogs = [
    e.catalog.lower()
    for e in spark.sql(f"SHOW CATALOGS").collect()
    if e.catalog not in ["main", "samples", "system", "__databricks_internal"]
]
print(f"Catalogs in the workspace: {workspace_catalogs}")
banfield_catalogs = ["banfield_catalogdev", "banfield_catalogtst", "banfield_catalog"]
target_catalog0 = [e for e in banfield_catalogs if e in workspace_catalogs]
if (not target_catalog0) or len(target_catalog0) != 1:
    raise Exception(
        f"Expecting any one of the active banfield catalog but received {len(target_catalog0)}; Banfield catalog names: {banfield_catalogs}"
    )
spark.conf.set("catlg.banfield_catalog", target_catalog0[0])
print(f"catlg.banfield_catalog: {target_catalog0[0]}")
 
bf_vwmvhcores = ["bf_vwmvhcoredev", "bf_vwmvhcoretst", "bf_vwmvhcore"]
target_catalog1 = [e for e in bf_vwmvhcores if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog1) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog1)}; bf_vwmvhcore names: {bf_vwmvhcores}"
    )
spark.conf.set("catlg.bf_vwmvhcore", target_catalog1[0])
print(f"catlg.bf_vwmvhcore: {target_catalog1[0]}")


bf_vwedhs = ["bf_vwedhdev", "bf_vwedhtst", "bf_vwedh"]
target_catalog2 = [e for e in bf_vwedhs if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog2) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog2)}; bf_vwmvhcore names: {bf_vwedhs}"
    )
spark.conf.set("catlg.bf_vwedh", target_catalog2[0])
print(f"catlg.bf_vwedh: {target_catalog2[0]}")

bf_vwvoyagers = ["bf_vwvoyagerdev", "bf_vwvoyagertst", "bf_vwvoyager"]
target_catalog3 = [e for e in bf_vwvoyagers if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog3) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog3)}; bf_vwvoyager names: {bf_vwvoyagers}"
    )
spark.conf.set("catlg.bf_vwvoyager", target_catalog3[0])
print(f"catlg.bf_vwvoyager: {target_catalog3[0]}")

3. Create table Bronze Layer

In [0]:
%sql
create or replace table ${catlg.banfield_catalog}.bfdw_bronze.cmn_tbcmdpymttyp (
  dw_pymnt_typ_id bigint not null comment 'name: data warehouse payment type identifier
description: the surrogate key generated during the data warehouse load process to uniquely identify a payment type dimension record.
source: data warehouse unique sequence generator',

pymnt_typ_descr string comment 'name: payment type description
description: description of payment type
source: petware
dwpetnet.receiptpaymentmethod.receiptpaymentmethoddesc
paws
dwwpa.wp_account_type.account_type',

hosp_day_summary_tab_col_nam string comment 'name: hospital day summary table column name
description: name of column in tb_cmf_hosp_day_summary, where this payment type would be summarized per hospital.
if none, then use "n/a"
source: data warehouse
hard coded when new payment type comes in',

dw_create_dt timestamp not null comment 'name: data warehouse create date
description: dw metadata. the date that the record was first added to this table in the data warehouse.
this is the starting date/time of the process that created the record.
source: data warehouse local system date/time (us-pacific time zone).',

dw_begin_eff_dt timestamp not null comment 'name: data warehouse begin effective date
description: dw metadata. the effective start date/time for this record, or when this record first came into effect.
this only applies to tables that track historical changes. for others, this is hard coded to 1/1/1900 00:00:00
if this is the first instance of a given key, the value will be 1/1/1900 00:00:00
for any other instance of a given key, this value will be one second after the effective end date of the previous instance.
the one second difference is in there, so you can say mydate between begin eff date and end eff date and not worry about overlap.
example
key dw_begin_eff_dt dw_end_eff_dt dw_load_dt dw_curr_row_ind
abc 1/1/1900 00:00:00 3/4/2011 09:23:58 2/1/1998 05:31:43 0
abc 3/4/2011 09:23:59 12/31/9999 23:59:59 3/4/2011 09:23:58 1
source: data warehouse load process',

dw_end_eff_dt timestamp not null comment 'name: data warehouse end effective date
description: dw metadata. the effective end date/time for this record, or when this record stopped being in effect.
this only applies to tables that track historical changes. for others, this is hard coded to 12/31/9999 23:59:59
if this is the last instance of a given key, the value will be 12/31/9999 23:59:59
for any other instance of a given key, this value will be the date/time when the following instance is inserted, minus 1 second.
the one second difference is in there, so you can say mydate between begin eff date and end eff date and not worry about overlap.
example
key dw_begin_eff_dt dw_end_eff_dt dw_load_dt dw_curr_row_ind
abc 1/1/1900 00:00:00 3/4/2011 09:23:58 2/1/1998 05:31:43 0
abc 3/4/2011 09:23:59 12/31/9999 23:59:59 3/4/2011 09:23:58 1
source: data warehouse',

dw_deleted_ind int not null comment 'name: data warehouse deleted indicator
description: dw metadata. an indicator set to 1 to designate when a particular key has been removed from a source system of record.
a value of 0 indicates that the key is still present in the source system of record.
source: derived by dw load process, by comparing source keys against target keys.',

dw_curr_row_ind int not null comment 'name: data warehouse current row indicator
description: dw metadata. an indicator set to 0 in cases where this record is not the most recent version of the given key. it will be set to 1 for the record that is the most recent version for a given key.
this only applies to tables that track history. if this table does not track history, then this will be hard coded to 1
note, if the key has been removed from the source system of record, this indicator will still be set to 1.
source: data warehouse load process',

dw_job_id bigint not null comment 'name: data warehouse job identifier
description: dw metadata. an identifier for the data warehouse load process that last affected the record.
source: data warehouse database sequence generator' ,

dw_load_dt timestamp not null comment 'name: data warehouse load date
description: dw metadata. the date the data warehouse load process that last affected the record.
this is the starting date/time of the process that last inserted or updated the record.
source: data warehouse local system date/time (us-pacific time zone).',

fw_createdts timestamp  comment 'the timestamp when the record was first loaded into the iron table',
fw_modifiedts timestamp  comment 'the timestamp when the record was modified and loaded in bronze table',
fw_filename string comment 'the name of the file from which the record was loaded',
rowhash string comment 'hash value generated from the contents of each row, used for detecting changes, deduplication, and maintaining data integrity in Delta Lake tables',
record_status string comment 'reflects the status of the bronze layer record, such as valid or invalid. we will push only valid records to silver layer',
constraint `cmn_tbcmdpymttyp_pk` primary key (`dw_pymnt_typ_id`) rely)
using delta
comment 'name: inventory item dimension
description: the set of distinct items and services that may be referenced as a line item on an invoice.  this includes taxes, coupons and adjustments, as well as products and services.
grain: one row per inventory item, per historical change for that item
each inventory item will have multiple rows in this table. with a row for each change to any of its attributes.
each historical instance of a row will have an effective start and end date
the most recent version of a row is marked with a dw_curr_row_ind = 1
this table is loaded every day. note that currently, pmri is updated only every few weeks or so. in between releases, it is unlikely that any changes will occur in this table.
source: petware pmri
dwpetnet.inventory'
tblproperties (
  'delta.checkpoint.writestatsasjson' = 'false',
  'delta.checkpoint.writestatsasstruct' = 'true',
  'delta.minreaderversion' = '1',
  'delta.minwriterversion' = '2',
  'delta.feature.allowColumnDefaults' = 'supported')

4. Create table for Silver Layer

In [0]:
%sql
create or replace table ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdpymttyp (
  dw_pymnt_typ_id bigint not null comment 'name: data warehouse payment type identifier
description: the surrogate key generated during the data warehouse load process to uniquely identify a payment type dimension record.
source: data warehouse unique sequence generator',

pymnt_typ_descr string comment 'name: payment type description
description: description of payment type
source: petware
dwpetnet.receiptpaymentmethod.receiptpaymentmethoddesc
paws
dwwpa.wp_account_type.account_type',

hosp_day_summary_tab_col_nam string comment 'name: hospital day summary table column name
description: name of column in tb_cmf_hosp_day_summary, where this payment type would be summarized per hospital.
if none, then use "n/a"
source: data warehouse
hard coded when new payment type comes in',

dw_create_dt timestamp not null comment 'name: data warehouse create date
description: dw metadata. the date that the record was first added to this table in the data warehouse.
this is the starting date/time of the process that created the record.
source: data warehouse local system date/time (us-pacific time zone).',

dw_begin_eff_dt timestamp not null comment 'name: data warehouse begin effective date
description: dw metadata. the effective start date/time for this record, or when this record first came into effect.
this only applies to tables that track historical changes. for others, this is hard coded to 1/1/1900 00:00:00
if this is the first instance of a given key, the value will be 1/1/1900 00:00:00
for any other instance of a given key, this value will be one second after the effective end date of the previous instance.
the one second difference is in there, so you can say mydate between begin eff date and end eff date and not worry about overlap.
example
key dw_begin_eff_dt dw_end_eff_dt dw_load_dt dw_curr_row_ind
abc 1/1/1900 00:00:00 3/4/2011 09:23:58 2/1/1998 05:31:43 0
abc 3/4/2011 09:23:59 12/31/9999 23:59:59 3/4/2011 09:23:58 1
source: data warehouse load process',

dw_end_eff_dt timestamp not null comment 'name: data warehouse end effective date
description: dw metadata. the effective end date/time for this record, or when this record stopped being in effect.
this only applies to tables that track historical changes. for others, this is hard coded to 12/31/9999 23:59:59
if this is the last instance of a given key, the value will be 12/31/9999 23:59:59
for any other instance of a given key, this value will be the date/time when the following instance is inserted, minus 1 second.
the one second difference is in there, so you can say mydate between begin eff date and end eff date and not worry about overlap.
example
key dw_begin_eff_dt dw_end_eff_dt dw_load_dt dw_curr_row_ind
abc 1/1/1900 00:00:00 3/4/2011 09:23:58 2/1/1998 05:31:43 0
abc 3/4/2011 09:23:59 12/31/9999 23:59:59 3/4/2011 09:23:58 1
source: data warehouse',

dw_deleted_ind int not null comment 'name: data warehouse deleted indicator
description: dw metadata. an indicator set to 1 to designate when a particular key has been removed from a source system of record.
a value of 0 indicates that the key is still present in the source system of record.
source: derived by dw load process, by comparing source keys against target keys.',

dw_curr_row_ind int not null comment 'name: data warehouse current row indicator
description: dw metadata. an indicator set to 0 in cases where this record is not the most recent version of the given key. it will be set to 1 for the record that is the most recent version for a given key.
this only applies to tables that track history. if this table does not track history, then this will be hard coded to 1
note, if the key has been removed from the source system of record, this indicator will still be set to 1.
source: data warehouse load process',

dw_job_id bigint not null comment 'name: data warehouse job identifier
description: dw metadata. an identifier for the data warehouse load process that last affected the record.
source: data warehouse database sequence generator' ,

dw_load_dt timestamp not null comment 'name: data warehouse load date
description: dw metadata. the date the data warehouse load process that last affected the record.
this is the starting date/time of the process that last inserted or updated the record.
source: data warehouse local system date/time (us-pacific time zone).',

fw_createdts timestamp  comment 'the timestamp when the record was first loaded into the iron table',
fw_modifiedts timestamp  comment 'the timestamp when the record was modified and loaded in bronze table',
fw_filename string comment 'the name of the file from which the record was loaded',
rowhash string comment 'hash value generated from the contents of each row, used for detecting changes, deduplication, and maintaining data integrity in Delta Lake tables',
record_status string comment 'reflects the status of the bronze layer record, such as valid or invalid. we will push only valid records to silver layer',
constraint `cmn_tbcmdpymttyp_pk` primary key (`dw_pymnt_typ_id`) rely)
using delta
comment 'name: inventory item dimension
description: the set of distinct items and services that may be referenced as a line item on an invoice.  this includes taxes, coupons and adjustments, as well as products and services.
grain: one row per inventory item, per historical change for that item
each inventory item will have multiple rows in this table. with a row for each change to any of its attributes.
each historical instance of a row will have an effective start and end date
the most recent version of a row is marked with a dw_curr_row_ind = 1
this table is loaded every day. note that currently, pmri is updated only every few weeks or so. in between releases, it is unlikely that any changes will occur in this table.
source: petware pmri
dwpetnet.inventory'
tblproperties (
  'delta.checkpoint.writestatsasjson' = 'false',
  'delta.checkpoint.writestatsasstruct' = 'true',
  'delta.minreaderversion' = '1',
  'delta.minwriterversion' = '2',
  'delta.feature.allowColumnDefaults' = 'supported')

5. Create View for Gold Layer

In [0]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_gold.cmn_tbcmdpymttyp

as

select a.*
from ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdpymttyp a where 1 =1 

6. Create View for bfdw_date_quality for silver layer

In [0]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_data_quality.cmn_tbcmdpymttyp_silver_primary_key_exceptions

as

select a.*
from ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdpymttyp a where 1 !=1 


7. Create View for bfdw_data_quality for Bronze layers

In [0]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_data_quality.cmn_tbcmdpymttyp_bronze_record_status_exceptions

as
Select *  
from  ${catlg.banfield_catalog}.bfdw_bronze.cmn_tbcmdpymttyp
where record_status != 'valid';